<a href="https://colab.research.google.com/github/JagdishMane/numpy-pytorch-tensorflow/blob/main/Advertising_ANN_Classification_Notebook_Jagdish_Mane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2.1: Building an Artificail Neural Network

### Predicting Online Ad Clicks Using an Artificial Neural Network

A classification model to predict whether a user will click on an online advertisement using demographic and behavioral attributes. This Artificial Neural Network ANN built with TensorFlow.

1. This Notebook is tested on Google Colab.
2. Ensure to upload the dataset file "M2-Advertising-Dataset.csv"
3. Some part of Code has been generated using Google Colab compose subagent inbuildint in Colab Tools.

## 1. Load Dataset into Pandas dataframe

In [3]:
import numpy as np
import pandas as pd

data_path = "M2-Advertising_Dataset.csv"
df = pd.read_csv(data_path)


print("Shape:", df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'M2-Advertising_Dataset.csv'

In [2]:
##### TO BE REMOVED - Reference only
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

DATA_PATH = "M2-Advertising_Dataset.csv"

KeyboardInterrupt: 

## Task 1 & 2: Load Dataset, Data Cleaning, and Initial Exploration

The dataset contains user browsing behavior, demographic details, and the target variable `Clicked on Ad`.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nFirst five rows:")
display(df.head())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["Clicked on Ad"].value_counts())

print("\nSummary statistics:")
display(df.describe())

### Data Quality Summary

The original report states that the dataset contains:

- 1,000 records
- 10 columns
- No missing values
- No duplicate rows
- A balanced target distribution: 500 clicked and 500 not clicked

Because the target is balanced, accuracy is a meaningful metric and class-imbalance correction is not required.

## Preprocessing and Feature Engineering

Some columns need special handling before training a numeric ANN model:

- `Ad Topic Line`: free-text field with very high uniqueness; dropped because it needs separate NLP feature extraction.
- `City`: very high cardinality; dropped to avoid sparse one-hot encoding and overfitting.
- `Country`: frequency encoded as `Country_Freq`.
- `Timestamp`: converted into `Hour`, `DayOfWeek`, and `Month`.

In [ ]:
# Convert Timestamp into datetime features
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

df["Hour"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month

# Frequency encode Country
country_freq = df["Country"].value_counts(normalize=True)
df["Country_Freq"] = df["Country"].map(country_freq)

# Drop high-cardinality/free-text/raw timestamp columns
drop_cols = ["Ad Topic Line", "City", "Country", "Timestamp"]
df_model = df.drop(columns=drop_cols)

print("Modeling dataset shape:", df_model.shape)
display(df_model.head())

print("\nModeling columns:")
print(df_model.columns.tolist())

## Exploratory Data Analysis

The following plots show the distribution of key numeric predictors split by ad-click outcome.

In [ ]:
numeric_features = [
    "Daily Time Spent on Site",
    "Age",
    "Area Income",
    "Daily Internet Usage"
]

for feature in numeric_features:
    plt.figure(figsize=(7, 4))
    for label in sorted(df["Clicked on Ad"].unique()):
        subset = df[df["Clicked on Ad"] == label][feature]
        plt.hist(subset, bins=25, alpha=0.6, label=f"Clicked on Ad = {label}")
    plt.title(f"Distribution of {feature} by Click Outcome")
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.legend()
    plt.show()

### Correlation Analysis

This heatmap shows how strongly each modeling feature is linearly related to the target variable.

In [ ]:
corr = df_model.corr(numeric_only=True)

plt.figure(figsize=(10, 7))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

target_corr = corr["Clicked on Ad"].sort_values(ascending=False)
print("Feature correlation with target 'Clicked on Ad':")
print(target_corr)

## Task 3 & 4: Train/Test Split and Feature Scaling

The processed data is split into:

- 80% training data
- 20% test data

Stratified sampling is used to preserve the balanced target distribution in both sets.

`StandardScaler` is fitted only on the training data and then applied to both training and test data to avoid data leakage.

In [ ]:
X = df_model.drop(columns=["Clicked on Ad"])
y = df_model["Clicked on Ad"]

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True))
print("y_test distribution:")
print(y_test.value_counts(normalize=True))

## Task 5 & 6: Build and Tune the ANN Classifier

Five ANN configurations are tested. Each model uses:

- Binary cross-entropy loss
- Adam optimizer
- Sigmoid output layer
- Early stopping based on validation loss

In [ ]:
n_features = X_train_scaled.shape[1]

def build_model(hidden_layers=(16, 8), activation="relu", dropout=0.0, l2=0.0, lr=0.001):
    reg = keras.regularizers.l2(l2) if l2 > 0 else None

    model = keras.Sequential()
    model.add(layers.Input(shape=(n_features,)))

    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation, kernel_regularizer=reg))
        if dropout > 0:
            model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
experiments = [
    {
        "name": "Exp1: [16, 8], ReLU, no regularization",
        "hidden_layers": (16, 8),
        "activation": "relu",
        "dropout": 0.0,
        "l2": 0.0,
        "lr": 0.001,
        "batch_size": 32
    },
    {
        "name": "Exp2: [32, 16, 8], ReLU, dropout 0.2",
        "hidden_layers": (32, 16, 8),
        "activation": "relu",
        "dropout": 0.2,
        "l2": 0.0,
        "lr": 0.001,
        "batch_size": 32
    },
    {
        "name": "Exp3: [32, 16], Tanh, lr 0.01",
        "hidden_layers": (32, 16),
        "activation": "tanh",
        "dropout": 0.0,
        "l2": 0.0,
        "lr": 0.01,
        "batch_size": 16
    },
    {
        "name": "Exp4: [64, 32, 16], ReLU, dropout 0.3 + L2",
        "hidden_layers": (64, 32, 16),
        "activation": "relu",
        "dropout": 0.3,
        "l2": 0.001,
        "lr": 0.0005,
        "batch_size": 64
    },
    {
        "name": "Exp5: [16], ReLU, simple model",
        "hidden_layers": (16,),
        "activation": "relu",
        "dropout": 0.0,
        "l2": 0.0,
        "lr": 0.001,
        "batch_size": 32
    }
]

pd.DataFrame(experiments)

In [ ]:
results = []
histories = {}
trained_models = {}

for exp in experiments:
    print(f"\nTraining {exp['name']}")

    tf.keras.backend.clear_session()
    tf.random.set_seed(RANDOM_STATE)

    model = build_model(
        hidden_layers=exp["hidden_layers"],
        activation=exp["activation"],
        dropout=exp["dropout"],
        l2=exp["l2"],
        lr=exp["lr"]
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.2,
        epochs=200,
        batch_size=exp["batch_size"],
        callbacks=[early_stop],
        verbose=0
    )

    probs = model.predict(X_test_scaled, verbose=0).ravel()
    preds = (probs >= 0.5).astype(int)

    row = {
        "name": exp["name"],
        "epochs_trained": len(history.history["loss"]),
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs)
    }

    results.append(row)
    histories[exp["name"]] = history
    trained_models[exp["name"]] = model

    print(row)

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
display(results_df)

## Experiment Comparison and Final Model Selection

The best model is selected using ROC AUC as the primary ranking metric.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["name"], results_df["roc_auc"])
plt.title("ANN Experiment Comparison by ROC AUC")
plt.ylabel("ROC AUC")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

best_name = results_df.iloc[0]["name"]
best_model = trained_models[best_name]
best_history = histories[best_name]

print("Best model:", best_name)
display(results_df.iloc[[0]])

## Task 7: Evaluate the Final ANN Model

The final ANN model is evaluated on the held-out test set using:

- Accuracy
- Precision
- Recall
- F1 score
- ROC AUC
- Confusion matrix
- ROC curve

In [ ]:
probs = best_model.predict(X_test_scaled, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)

print(classification_report(y_test, preds, target_names=["No Click", "Clicked"]))

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"],
    "Score": [
        accuracy_score(y_test, preds),
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds),
        roc_auc_score(y_test, probs)
    ]
})

display(metrics_df)

In [ ]:
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["No Click", "Clicked"])
plt.yticks([0, 1], ["No Click", "Clicked"])
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, probs)
auc_value = roc_auc_score(y_test, probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC AUC = {auc_value:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(best_history.history["accuracy"], label="Training Accuracy")
plt.plot(best_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## Task 8: Feature Importance and Interpretation

Neural networks do not directly provide built-in feature importance scores like tree-based models.

Here, permutation importance is used:

1. Measure baseline test accuracy.
2. Shuffle one feature column at a time.
3. Measure how much accuracy drops.
4. Larger accuracy drop means the model relies more on that feature.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

baseline_acc = accuracy_score(y_test, preds)
n_repeats = 30

importances = np.zeros((len(feature_names), n_repeats))

for col_idx, feature in enumerate(feature_names):
    for rep in range(n_repeats):
        X_permuted = X_test_scaled.copy()
        rng.shuffle(X_permuted[:, col_idx])

        permuted_probs = best_model.predict(X_permuted, verbose=0).ravel()
        permuted_preds = (permuted_probs >= 0.5).astype(int)
        permuted_acc = accuracy_score(y_test, permuted_preds)

        importances[col_idx, rep] = baseline_acc - permuted_acc

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Accuracy_Drop": importances.mean(axis=1),
    "Std_Accuracy_Drop": importances.std(axis=1)
}).sort_values("Mean_Accuracy_Drop", ascending=False)

display(importance_df)

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(importance_df["Feature"], importance_df["Mean_Accuracy_Drop"])
plt.gca().invert_yaxis()
plt.title("Permutation Feature Importance")
plt.xlabel("Mean Accuracy Drop")
plt.tight_layout()
plt.show()

## Conclusions

Based on the original report logic, the strongest predictors are expected to be:

- `Daily Internet Usage`
- `Daily Time Spent on Site`
- `Area Income`
- `Age`

The original report concluded that behavioral variables were the most important signals, while gender, country frequency, and temporal features had much weaker influence.

# Bonus: Comparison with Classical Machine Learning Models

This section compares the ANN with four classical models:

- Logistic Regression
- Decision Tree
- Random Forest
- SVM with RBF kernel

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

classical_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE),
    "SVM (RBF kernel)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
}

classical_results = []

for name, clf in classical_models.items():
    clf.fit(X_train_scaled, y_train)

    if hasattr(clf, "predict_proba"):
        model_probs = clf.predict_proba(X_test_scaled)[:, 1]
    else:
        model_probs = clf.decision_function(X_test_scaled)

    model_preds = clf.predict(X_test_scaled)

    classical_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, model_preds),
        "Precision": precision_score(y_test, model_preds),
        "Recall": recall_score(y_test, model_preds),
        "F1": f1_score(y_test, model_preds),
        "ROC AUC": roc_auc_score(y_test, model_probs)
    })

# Add ANN result for comparison
classical_results.append({
    "Model": "ANN",
    "Accuracy": accuracy_score(y_test, preds),
    "Precision": precision_score(y_test, preds),
    "Recall": recall_score(y_test, preds),
    "F1": f1_score(y_test, preds),
    "ROC AUC": roc_auc_score(y_test, probs)
})

model_comparison_df = pd.DataFrame(classical_results).sort_values("ROC AUC", ascending=False)
display(model_comparison_df)

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(model_comparison_df["Model"], model_comparison_df["ROC AUC"])
plt.title("Model Comparison by ROC AUC")
plt.ylabel("ROC AUC")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Random Forest Feature Importance Cross-Check

Random Forest provides built-in feature importance. This can be used as an independent check against ANN permutation importance.

In [ ]:
rf_model = classical_models["Random Forest"]

rf_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(rf_importance_df)

plt.figure(figsize=(9, 5))
plt.barh(rf_importance_df["Feature"], rf_importance_df["Importance"])
plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## Final Notes

The original Word-report generation script has been converted into a notebook-style workflow.

This notebook keeps the same project structure:

1. Load and inspect data
2. Clean and preprocess
3. Engineer features
4. Split and scale data
5. Train ANN models
6. Select the best ANN
7. Evaluate final model
8. Interpret feature importance
9. Compare with classical ML models